# Optimization Landscape — Exercises

This notebook contains 8 exercises covering critical-point classification, Hessian spectra, symmetry, overparameterization, sharpness, interpolation barriers, edge-of-stability intuition, and practical landscape diagnosis.

**Difficulty:** Exercises 1-3 are mechanical, 4-6 are conceptual/theoretical, and 7-8 are applied ML interpretation exercises.


In [ ]:
import numpy as np

def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_close(name, value, target, tol=1e-8):
    ok = np.allclose(value, target, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    if not ok:
        print("  value :", value)
        print("  target:", target)
    return ok

def check_true(name, condition):
    print(f"{'PASS' if condition else 'FAIL'} - {name}")
    return condition

def loss_uv(theta):
    u, v = theta
    return 0.5 * (u * v - 1.0) ** 2

def hess_uv(theta):
    u, v = theta
    return np.array([
        [v**2, 2 * u * v - 1.0],
        [2 * u * v - 1.0, u**2],
    ], dtype=float)

def line_path(a, b, ts):
    return np.array([(1 - t) * np.array(a) + t * np.array(b) for t in ts])

def path_barrier(path, loss_fn):
    vals = np.array([loss_fn(p) for p in path])
    return vals.max() - max(vals[0], vals[-1]), vals

print("Exercise helpers ready.")


---

## Exercise 1 [*] — Classify a saddle and a minimum

For the objective

$$
\mathcal{L}(u,v) = \frac{1}{2}(uv - 1)^2,
$$

classify the points $(0,0)$ and $(1,1)$ using the Hessian eigenvalues.


In [ ]:
# Exercise 1: Your Solution
theta_s = np.array([0.0, 0.0])
theta_m = np.array([1.0, 1.0])
pass


In [ ]:
# Exercise 1: Solution
import numpy as np

def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_true(name, condition):
    print(f"{'PASS' if condition else 'FAIL'} - {name}")
    return condition

def hess_uv(theta):
    u, v = theta
    return np.array([
        [v**2, 2 * u * v - 1.0],
        [2 * u * v - 1.0, u**2],
    ], dtype=float)

header("Exercise 1")
eig_s = np.linalg.eigvalsh(hess_uv(np.array([0.0, 0.0])))
eig_m = np.linalg.eigvalsh(hess_uv(np.array([1.0, 1.0])))
print("Saddle eigenvalues :", eig_s)
print("Minimum eigenvalues:", eig_m)
check_true("Origin is a strict saddle", eig_s[0] < 0 < eig_s[1])
check_true("(1,1) is a degenerate minimum", eig_m[0] >= -1e-12 and eig_m[1] > 0)
print("\nTakeaway: the same loss can contain both negative-curvature saddles and flat minima.")


---

## Exercise 2 [*] — Hessian signature

Let

$$
H =
\begin{bmatrix}
3 & 1 \\
1 & -2
\end{bmatrix}.
$$

Compute its eigenvalues and decide whether it corresponds locally to a minimum, maximum, or saddle.


In [ ]:
# Exercise 2: Your Solution
H = np.array([[3.0, 1.0], [1.0, -2.0]])
pass


In [ ]:
# Exercise 2: Solution
import numpy as np

def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_true(name, condition):
    print(f"{'PASS' if condition else 'FAIL'} - {name}")
    return condition

header("Exercise 2")
H = np.array([[3.0, 1.0], [1.0, -2.0]])
eigs = np.linalg.eigvalsh(H)
print("Eigenvalues:", eigs)
check_true("Mixed-sign spectrum implies a saddle", eigs[0] < 0 < eigs[1])
print("\nTakeaway: Hessian signs classify local geometry much more strongly than gradient norm alone.")


---

## Exercise 3 [*] — Symmetry-preserving rescaling

Show numerically that the points $(1,1)$ and $(3,\frac{1}{3})$ define the same scalar function under the model $x \mapsto uvx$.


In [ ]:
# Exercise 3: Your Solution
xs = np.linspace(-2, 2, 20)
pass


In [ ]:
# Exercise 3: Solution
import numpy as np

def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_close(name, value, target, tol=1e-8):
    ok = np.allclose(value, target, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    return ok

header("Exercise 3")
xs = np.linspace(-2, 2, 20)
y1 = (1.0 * 1.0) * xs
y2 = (3.0 * (1.0 / 3.0)) * xs
check_close("Outputs agree on all sampled inputs", y1, y2)
print("\nTakeaway: parameter-space distance can hide exact function-space equivalence.")


---

## Exercise 4 [**] — Interpolation barrier

Measure the straight-line interpolation barrier between $(0.5,2)$ and $(2,0.5)$ for the loss

$$
\mathcal{L}(u,v) = \frac{1}{2}(uv - 1)^2.
$$


In [ ]:
# Exercise 4: Your Solution
theta_a = np.array([0.5, 2.0])
theta_b = np.array([2.0, 0.5])
pass


In [ ]:
# Exercise 4: Solution
import numpy as np

def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_true(name, condition):
    print(f"{'PASS' if condition else 'FAIL'} - {name}")
    return condition

def loss_uv(theta):
    u, v = theta
    return 0.5 * (u * v - 1.0) ** 2

header("Exercise 4")
theta_a = np.array([0.5, 2.0])
theta_b = np.array([2.0, 0.5])
ts = np.linspace(0, 1, 401)
path = np.array([(1 - t) * theta_a + t * theta_b for t in ts])
vals = np.array([loss_uv(p) for p in path])
barrier = vals.max() - max(vals[0], vals[-1])
print("Barrier =", barrier)
check_true("The straight line rises above zero", barrier > 0.05)
print("\nTakeaway: a linear barrier does not rule out a nonlinear low-loss connection.")


---

## Exercise 5 [**] — Overparameterized minimum

For

$$
\mathcal{L}(a,b,c) = \frac{1}{2}(abc - 1)^2,
$$

explain why a point with $abc=1$ should have at least two flat directions.


In [ ]:
# Exercise 5: Your Solution
H_guess = np.ones((3, 3))
pass


In [ ]:
# Exercise 5: Solution
import numpy as np

def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_true(name, condition):
    print(f"{'PASS' if condition else 'FAIL'} - {name}")
    return condition

header("Exercise 5")
H = np.ones((3, 3))
eigs = np.linalg.eigvalsh(H)
print("Eigenvalues:", eigs)
check_true("Two eigenvalues are zero", np.sum(np.abs(eigs) < 1e-12) == 2)
print("\nTakeaway: one scalar constraint on three parameters leaves a two-dimensional solution manifold.")


---

## Exercise 6 [**] — Sharpness caveat

Compare the top Hessian eigenvalue at $(1,1)$ and $(3,\frac{1}{3})$. What does the comparison say about naive flatness claims?


In [ ]:
# Exercise 6: Your Solution
theta_bal = np.array([1.0, 1.0])
theta_unbal = np.array([3.0, 1.0 / 3.0])
pass


In [ ]:
# Exercise 6: Solution
import numpy as np

def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_true(name, condition):
    print(f"{'PASS' if condition else 'FAIL'} - {name}")
    return condition

def hess_uv(theta):
    u, v = theta
    return np.array([
        [v**2, 2 * u * v - 1.0],
        [2 * u * v - 1.0, u**2],
    ], dtype=float)

header("Exercise 6")
lam_bal = np.max(np.linalg.eigvalsh(hess_uv(np.array([1.0, 1.0]))))
lam_unbal = np.max(np.linalg.eigvalsh(hess_uv(np.array([3.0, 1.0 / 3.0]))))
print("lambda_max balanced  =", lam_bal)
print("lambda_max unbalanced=", lam_unbal)
check_true("Same function can have different raw sharpness", lam_unbal > lam_bal * 2)
print("\nTakeaway: raw parameter-space sharpness is not invariant under function-preserving reparameterization.")


---

## Exercise 7 [***] — Stability threshold

For the quadratic loss

$$
f(\boldsymbol{\theta}) = \frac{1}{2}\boldsymbol{\theta}^\top
\begin{bmatrix}
1 & 0 \\
0 & 10
\end{bmatrix}
\boldsymbol{\theta},
$$

what is the largest stable learning rate for plain gradient descent?


In [ ]:
# Exercise 7: Your Solution
H = np.diag([1.0, 10.0])
pass


In [ ]:
# Exercise 7: Solution
import numpy as np

def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_close(name, value, target, tol=1e-8):
    ok = np.allclose(value, target, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    return ok

header("Exercise 7")
H = np.diag([1.0, 10.0])
eta_max = 2.0 / np.max(np.diag(H))
print("Largest stability threshold =", eta_max)
check_close("eta_max = 2/lambda_max", eta_max, 0.2)
print("\nTakeaway: top curvature sets the tightest local stability limit for quadratic GD.")


---

## Exercise 8 [***] — Landscape-aware checkpoint averaging

Explain, in practical ML terms, when checkpoint averaging should help and when it should fail.


In [ ]:
# Exercise 8: Your Solution
pass


In [ ]:
# Exercise 8: Solution
def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_true(name, condition):
    print(f"{'PASS' if condition else 'FAIL'} - {name}")
    return condition

header("Exercise 8")
print("Averaging helps when checkpoints lie in a compatible low-loss corridor or connected region.")
print("It fails when checkpoints are separated by symmetry mismatch, task mismatch, or high barriers.")
check_true("A correct answer mentions both geometry and compatibility", True)
print("\nTakeaway: averaging is a geometric assumption about the relationship between the checkpoints, not a free lunch.")
